# DisTFD vs. TANE on Abalone

This report checks whether DisTFD finds the same functional dependencies as TANE. The dataset used is UCI Abalone to mimic the paper's experiment on accuracy. DisTFD and TANE are run on PySpark with 4 local cores as 4 nodes, seeds 0–2. The ground truth is Desbordante's TANE. 

**Contents**

1. [DisTFD returns exactly TANE's FDs](#s1)
2. [The agreement holds at every LHS size](#s2)
3. [Sampling does not change the result](#s3)
4. [Non-minimal FDs from the top-down levels are filtered out](#s4)
5. [Comparison with the paper](#s5)
6. [Conclusion](#conclusion)
7. [Appendix: abbreviations](#appendix)

In [25]:
# Import necessary libraries
import json
from collections import Counter
from pathlib import Path
import pandas as pd

# Set up paths and retrieve results
RES = (Path("../results") if Path("../results").exists() else Path("results")) / "abalone"
acc = pd.read_csv(RES / "accuracy.csv")
sweep = pd.read_csv(RES / "sampling_sweep.csv")
fds = RES / "fds"
truth = [tuple(map(json.dumps, fd)) for fd in json.loads((fds / "abalone__TANE.json").read_text())]
found = {s: [tuple(map(json.dumps, fd)) for fd in json.loads((fds / f"abalone__DisTFD__seed{s}.json").read_text())]
         for s in acc.seed}

<a id="s1"></a>
## 1. DisTFD returns exactly TANE's FDs

Each instance of functional dependency is denoted as (X, A). The results from DisTFD and TANE is reported as following: 
* TP if both DisTFD and TANE report it, 
* FP if only DisTFD does, 
* FN if only TANE does.

In [26]:
print("disagreeing FDs per seed:", {s: len(set(f) ^ set(truth)) for s, f in found.items()})
acc[["seed", "n_truth", "n_fds", "tp", "fp", "fn", "precision", "recall", "f1"]]

disagreeing FDs per seed: {0: 0, 1: 0, 2: 0}


,seed,n_truth,n_fds,tp,fp,fn,precision,recall,f1
0,0,137,137,137,0,0,1.0,1.0,1.0
1,1,137,137,137,0,0,1.0,1.0,1.0
2,2,137,137,137,0,0,1.0,1.0,1.0


On all 3 seeds **DisTFD reports the same** 137 minimal FDs as TANE: 137 TP, 0 FP, 0 FN, so precision, recall and F1 are 1.0.

<a id="s2"></a>
## 2. The agreement holds at every LHS size

In [27]:
size = lambda fd: len(json.loads(fd[0]))
by_size = pd.DataFrame({"TANE": Counter(map(size, truth)),
                        **{f"DisTFD seed {s}": Counter(map(size, f)) for s, f in found.items()}}).sort_index()
by_size.index.name = "|LHS|"
by_size

,TANE,DisTFD seed 0,DisTFD seed 1,DisTFD seed 2
|LHS|,,,,
3,18,18,18,18
4,79,79,79,79
5,32,32,32,32
6,8,8,8,8


TANE's FDs have 3 to 6 attributes on the left-hand side (18, 79, 32 and 8 FDs). **DisTFD has the same count** at every size on every seed, including the 8 FDs with a 6-attribute LHS.

<a id="s3"></a>
## 3. Sampling does not change the result

In [28]:
sweep.groupby("sample_ratio").agg(sample_rows=("sample_rows", "first"), share_pruned=("share_pruned", "mean"),
                                  f1_min=("f1", "min")).round(4)

,sample_rows,share_pruned,f1_min
sample_ratio,,,
0.05,208,0.2858,1.0
0.10,417,0.3905,1.0
0.25,1044,0.5213,1.0


At sample ratios of 5%, 10% and 25%, sampling rejects on average 29%, 39% and 52% of the candidates before the distributed check, and **F1 is 1.0 on every seed**. This reflects the theory described in the paper.

<a id="s4"></a>
## 4. Non-minimal FDs from the top-down levels are filtered out

In [29]:
st = json.loads((fds / "abalone__DisTFD__seed0__stats.json").read_text())
pd.DataFrame(st["levels"])

,level,candidates,pruned_by_sample,validated,valid
0,1,72,72,0,0
1,8,9,0,9,9
2,2,252,228,24,0
3,7,72,0,72,72
4,3,504,252,252,18
5,6,144,0,144,120
6,4,316,36,280,79
7,5,119,9,110,32


Two-way traversal visits the levels in the order 1, 8, 2, 7, 3, 6, 4, 5 (seed 0 shown).

 Levels 8, 7 and 6 return 201 valid FDs, of which 8 are minimal. **DisTFD's final minimality check** removes the other 193, so **none appear as false positives**.

<a id="s5"></a>
## 5. Comparison with the paper

| | Paper | Reproduction |
|---|---|---|
| Data | Abalone, 4,177 × 9 | UCI Abalone, 4,177 × 9 |
| Ground truth | TANE | Desbordante TANE |
| Metrics | precision, recall, F1 | precision, recall, F1 |
| Nodes | 8-server cluster; count for this experiment not stated | 4 (Spark local mode) |
| Sample ratio | not stated | 10% (417–418 rows) |
| Skew threshold | not stated | 0.1 |
| FD counts | not reported | 137 |

In [30]:
ours = acc[["tp", "fp", "fn"]].sum()
p, r = ours.tp / (ours.tp + ours.fp), ours.tp / (ours.tp + ours.fn)
pd.DataFrame({"paper": [1.0, 0.9854, 0.9926], "ours": [p, r, 2 * p * r / (p + r)]},
             index=["precision", "recall", "F1"]).round(4)

,paper,ours
precision,1.0000,1.0
recall,0.9854,1.0
F1,0.9926,1.0


The reproduction uses the same data, the same ground-truth algorithm and the same metrics as the paper. Precision is 1 in both. Recall is 0.9854 in the paper and 1.0 here, and F1 is 0.9926 and 1.0. 

However, **the reproduction did not return the identical result as that from the paper**. The reason might lay in LHS limit, sample ratio, skew threshold, node count or FD counts; as Wu and Mao did not included these parameters for this experiment.

<a id="conclusion"></a>
## Conclusion

**DisTFD works as well as TANE on Abalone.** It returns the same 137 minimal FDs on every seed (1), at every LHS size (2), at every sample ratio tested (3), with no non-minimal FDs left over (4). Its precision matches the paper's, and its recall and F1 are higher (5).

However, I acknowledge the reproduction only compares DisTFD and TANE, and did not include Cet and HFDD baselines.

<a id="appendix"></a>
## Appendix: abbreviations

| Abbreviation | Meaning |
|---|---|
| Cet | Centralized baseline in the paper: all data is collected on one node and searched there |
| DisTFD | The distributed FD discovery algorithm of Wu & Mao (2022) |
| F1 | Harmonic mean of precision and recall, 2 · precision · recall / (precision + recall) |
| FD | Functional dependency X → A: rows that agree on the attributes X also agree on A |
| FN | False negative: an FD that TANE reports and DisTFD does not |
| FP | False positive: an FD that DisTFD reports and TANE does not |
| HFDD | Distributed FD discovery baseline of Tu & Huang (2016) |
| LHS | Left-hand side of an FD (X in X → A); \|LHS\| is its number of attributes |
| TANE | Exact level-wise FD discovery algorithm of Huhtala et al. (1999), used as the ground truth |
| TP | True positive: an FD that both DisTFD and TANE report |
| UCI | University of California, Irvine (Machine Learning Repository, source of Abalone) |